# Whisper ASR Fine-tuning using AutoTrain Advanced

In this notebook, we will fine-tune a Whisper model for Automatic Speech Recognition (ASR) using AutoTrain Advanced.
You can replace the model with any Hugging Face Whisper-compatible model and dataset with any other dataset in proper formatting.
For dataset formatting, please take a look at [docs](https://huggingface.co/docs/autotrain/index).

In [ ]:
from autotrain.trainers.asr import WhisperTrainingParams
from autotrain.trainers.asr import train_whisper

In [ ]:
HF_USERNAME = "your_huggingface_username"
HF_TOKEN = "your_huggingface_write_token" # get it from https://huggingface.co/settings/token
# It is recommended to use secrets or environment variables to store your HF_TOKEN
# your token is required if push_to_hub is set to True or if you are accessing a gated model/dataset

## Method 1: Using the direct training function

This method uses the `train_whisper` function directly, which gives you more control over the training process.

In [ ]:
# Configure training parameters
params = WhisperTrainingParams(
    model_name="openai/whisper-small",  # You can choose from tiny, base, small, medium, large-v2, large-v3
    language="en",  # Language code for ASR
    task="transcribe",  # "transcribe" or "translate"
    
    # Training parameters
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    mixed_precision="fp16",
    
    # PEFT/LoRA parameters for efficient fine-tuning
    use_peft=True,
    lora_r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    
    # Audio processing parameters
    sampling_rate=16000,
    max_duration_secs=30.0,
    preprocessing_num_workers=4,
    
    # Logging
    log="tensorboard"
)

In [ ]:
# Start training
train_whisper(
    params=params,
    dataset_path="mozilla-foundation/common_voice_11_0",  # HF dataset name or local path
    output_dir="whisper-finetuned",
    audio_column="audio",  # Column containing audio data
    text_column="sentence",  # Column containing transcription text
    push_to_hub=True,
    hub_model_id=f"{HF_USERNAME}/whisper-small-finetuned",
    hub_token=HF_TOKEN
)

## Method 2: Using a configuration file

You can also use a YAML configuration file to define your training parameters. This is useful for reproducibility and sharing configurations.

In [ ]:
%%writefile asr_config.yaml
task: speech-recognition
base_model: openai/whisper-small
project_name: autotrain-whisper-finetune
log: tensorboard
backend: local

data:
  path: mozilla-foundation/common_voice_11_0
  train_split: train
  valid_split: validation
  audio_column: audio
  text_column: sentence

params:
  sampling_rate: 16000
  max_duration_secs: 30.0
  language: en
  task: transcribe
  epochs: 3
  batch_size: 8
  learning_rate: 5e-5
  use_peft: true
  lora_r: 8
  lora_alpha: 32
  lora_dropout: 0.1
  mixed_precision: fp16

hub:
  username: ${HF_USERNAME}
  token: ${HF_TOKEN}
  push_to_hub: true

In [ ]:
# Run training using the config file
# Note: This would typically be run from the command line
# !autotrain --config asr_config.yaml

# For demonstration purposes, we'll show how to load and use the config in Python
import yaml
import os

# Set environment variables for the config
os.environ["HF_USERNAME"] = HF_USERNAME
os.environ["HF_TOKEN"] = HF_TOKEN

# Load the config
with open("asr_config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully!")
print(f"Model: {config['base_model']}")
print(f"Dataset: {config['data']['path']}")


## Using the fine-tuned model

Once training is complete, you can use your fine-tuned model for inference:

In [ ]:
from transformers import pipeline
import torch

# Replace with your model path or Hub ID
model_path = "whisper-finetuned"  # or f"{HF_USERNAME}/whisper-small-finetuned"

# Create ASR pipeline
asr = pipeline(
    "automatic-speech-recognition",
    model=model_path,
    torch_dtype=torch.float16,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Example usage with an audio file
# result = asr("path/to/audio.wav")
# print(result["text"])